# H&M Simple Final Project — LightGBM, TensorFlow, RandomForest

Uses the H&M ready dataset (built on median_version with feature engineering).

Same 3 algorithms, same settings, no changes. Just see if the new features improve accuracy.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_visible_devices(gpus[0], 'GPU')
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'TensorFlow GPU detected: {gpus[0]}')
else:
    print('No GPU detected for TensorFlow — will use CPU')
print(f'TensorFlow version: {tf.__version__}')

import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print('All imports ready.')

TensorFlow GPU detected: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
TensorFlow version: 2.10.0
All imports ready.


## 1. Load H&M ready dataset

In [2]:
DATA_DIR = r'A:\IDEs\PyCharm\hAnd-m-PersonalizedFashionRecommendations\H&M ready dataset'

print('Loading articles...')
articles = pd.read_parquet(os.path.join(DATA_DIR, 'articles.parquet'))
articles['article_id'] = articles['article_id'].astype(str)
print(f'  articles: {articles.shape}')

print('Loading customers...')
customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
customers['customer_id'] = customers['customer_id'].astype(str)
print(f'  customers: {customers.shape}')

print('Loading transactions (sampling 500K for speed)...')
transactions = pd.read_parquet(os.path.join(DATA_DIR, 'transactions.parquet'))
transactions['article_id'] = transactions['article_id'].astype(str)
transactions['customer_id'] = transactions['customer_id'].astype(str)
if len(transactions) > 500000:
    transactions = transactions.sample(n=500000, random_state=42)
print(f'  transactions sample: {transactions.shape}')

print('\nData loaded.')

Loading articles...
  articles: (105542, 17)
Loading customers...
  customers: (1371980, 14)
Loading transactions (sampling 500K for speed)...
  transactions sample: (500000, 9)

Data loaded.


## 2. Prepare features

Same setup as before: positive = real purchases, negative = random (customer, article) pairs.

Now using ALL available numeric + categorical features from the engineered dataset (except price and sales_channel_id to avoid leakage).

NO price, NO sales_channel_id — those leak the label.

In [3]:
# Identify usable features (exclude IDs, dates, price, channel, free text)
EXCLUDE = ['customer_id', 'article_id', 't_dat', 'price', 'sales_channel_id',
           'day_of_week', 'price_bracket']  # day_of_week is text, price_bracket is text

# Get numeric columns from customers
cust_numeric = [c for c in customers.columns if c not in EXCLUDE
                and customers[c].dtype in ['int64', 'float64', 'int32', 'float32']]
# Get numeric columns from articles
art_numeric = [c for c in articles.columns if c not in EXCLUDE
                and articles[c].dtype in ['int64', 'float64', 'int32', 'float32']]

# Get categorical columns (encode them)
cust_cat = [c for c in customers.columns if c not in EXCLUDE
            and customers[c].dtype == 'object']
art_cat = [c for c in articles.columns if c not in EXCLUDE
           and articles[c].dtype == 'object']

# Transaction numeric features (month, is_weekend)
tx_numeric = ['month', 'is_weekend']

ALL_NUMERIC = cust_numeric + art_numeric + tx_numeric
ALL_CATEGORICAL = cust_cat + art_cat

print(f'Numeric features ({len(ALL_NUMERIC)}):')
for f in ALL_NUMERIC:
    print(f'  {f}')
print(f'\nCategorical features ({len(ALL_CATEGORICAL)}):')
for f in ALL_CATEGORICAL:
    print(f'  {f}')

Numeric features (14):
  FN
  Active
  age
  customer_total_purchases
  customer_unique_articles
  price_avg_per_customer
  days_since_last_purchase
  repeat_purchase_rate
  product_type_no
  index_group_no
  article_total_purchases
  article_unique_customers
  month
  is_weekend

Categorical features (16):
  club_member_status
  fashion_news_frequency
  channel_preference
  customer_top_category
  product_type_name
  product_group_name
  graphical_appearance_name
  colour_group_name
  perceived_colour_value_name
  perceived_colour_master_name
  department_name
  index_code
  index_name
  index_group_name
  section_name
  garment_group_name


In [4]:
def prepare_dataset(articles_df, customers_df, transactions_df, n_samples=100000):
    """Build balanced binary classification dataset."""
    # Positive examples
    pos = transactions_df.sample(n=min(n_samples, len(transactions_df)), random_state=42)
    pos = pos[['customer_id', 'article_id', 'month', 'is_weekend']].copy()
    pos['label'] = 1

    # Negative examples
    all_article_ids = articles_df['article_id'].unique()
    customer_bought = transactions_df.groupby('customer_id')['article_id'].apply(set).to_dict()

    neg_rows = []
    neg_customers = np.random.choice(pos['customer_id'].unique(), size=len(pos), replace=True)
    for cid in neg_customers:
        bought = customer_bought.get(cid, set())
        rand_article = str(np.random.choice(all_article_ids))
        tries = 0
        while rand_article in bought and tries < 10:
            rand_article = str(np.random.choice(all_article_ids))
            tries += 1
        neg_rows.append({'customer_id': cid, 'article_id': rand_article,
                         'month': np.random.randint(1, 13), 'is_weekend': np.random.randint(0, 2),
                         'label': 0})

    neg = pd.DataFrame(neg_rows)
    data = pd.concat([pos, neg], ignore_index=True)

    # Merge customer features
    cust_cols = ['customer_id'] + [c for c in ALL_NUMERIC if c in customers_df.columns] + \
                [c for c in ALL_CATEGORICAL if c in customers_df.columns]
    data = data.merge(customers_df[cust_cols], on='customer_id', how='left')

    # Merge article features
    art_cols = ['article_id'] + [c for c in ALL_NUMERIC if c in articles_df.columns] + \
              [c for c in ALL_CATEGORICAL if c in articles_df.columns]
    data = data.merge(articles_df[art_cols], on='article_id', how='left')

    # Fill NaNs from merge
    for col in ALL_NUMERIC:
        if col in data.columns:
            data[col] = data[col].fillna(data[col].median() if data[col].notna().any() else 0)
    for col in ALL_CATEGORICAL:
        if col in data.columns:
            data[col] = data[col].fillna('UNKNOWN')

    # Encode categorical columns
    for col in ALL_CATEGORICAL:
        if col in data.columns:
            le = LabelEncoder()
            data[col] = le.fit_transform(data[col].astype(str))

    # Final feature set
    feature_cols = [c for c in ALL_NUMERIC + ALL_CATEGORICAL if c in data.columns]
    X = data[feature_cols].values
    y = data['label'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    return X_train, X_test, y_train, y_test, feature_cols

print('Prepare function ready.')

Prepare function ready.


## 3. Training functions — same as before, no changes

In [5]:
def train_lightgbm(X_train, y_train, X_test, y_test):
    try:
        model = lgb.LGBMClassifier(
            n_estimators=100, max_depth=10, learning_rate=0.1,
            device='gpu', random_state=42, verbose=-1
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        return accuracy_score(y_test, preds), 'GPU'
    except Exception as e:
        print(f'  LightGBM GPU failed ({str(e)[:60]}...), using CPU')
        model = lgb.LGBMClassifier(
            n_estimators=100, max_depth=10, learning_rate=0.1,
            random_state=42, verbose=-1
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        return accuracy_score(y_test, preds), 'CPU'


def train_tensorflow(X_train, y_train, X_test, y_test):
    input_dim = X_train.shape[1]
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=20, batch_size=512, verbose=0, validation_split=0.1)
    _, acc = model.evaluate(X_test, y_test, verbose=0)
    return acc, 'GPU' if gpus else 'CPU'


def train_randomforest(X_train, y_train, X_test, y_test):
    model = RandomForestClassifier(n_estimators=100, max_depth=15,
                                     random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds), 'CPU'

print('Training functions ready.')

Training functions ready.


## 4. Run all 3 algorithms on the H&M ready dataset

In [6]:
results = []

print(f'\n{"="*60}')
print(f'  DATASET: H&M ready (median_version + feature engineering)')
print(f'{"="*60}')

print('  Preparing features...')
X_train, X_test, y_train, y_test, feat_cols = prepare_dataset(articles, customers, transactions, n_samples=100000)
print(f'  Train: {X_train.shape}, Test: {X_test.shape}')
print(f'  Features ({len(feat_cols)}): {feat_cols}')

print('  Training LightGBM...')
lgb_acc, lgb_dev = train_lightgbm(X_train, y_train, X_test, y_test)
print(f'  LightGBM accuracy: {lgb_acc:.4f} ({lgb_dev})')

print('  Training TensorFlow NN...')
tf_acc, tf_dev = train_tensorflow(X_train, y_train, X_test, y_test)
print(f'  TensorFlow accuracy: {tf_acc:.4f} ({tf_dev})')

print('  Training RandomForest...')
rf_acc, rf_dev = train_randomforest(X_train, y_train, X_test, y_test)
print(f'  RandomForest accuracy: {rf_acc:.4f} ({rf_dev})')

results.append({
    'dataset': 'H&M ready (median+features)',
    'LightGBM': lgb_acc, 'LightGBM_device': lgb_dev,
    'TensorFlow': tf_acc, 'TensorFlow_device': tf_dev,
    'RandomForest': rf_acc, 'RandomForest_device': rf_dev,
})

print(f'\n{"="*60}')
print('All 3 runs complete.')
print(f'{"="*60}')


  DATASET: H&M ready (median_version + feature engineering)
  Preparing features...
  Train: (160000, 30), Test: (40000, 30)
  Features (30): ['FN', 'Active', 'age', 'customer_total_purchases', 'customer_unique_articles', 'price_avg_per_customer', 'days_since_last_purchase', 'repeat_purchase_rate', 'product_type_no', 'index_group_no', 'article_total_purchases', 'article_unique_customers', 'month', 'is_weekend', 'club_member_status', 'fashion_news_frequency', 'channel_preference', 'customer_top_category', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name', 'department_name', 'index_code', 'index_name', 'index_group_name', 'section_name', 'garment_group_name']
  Training LightGBM...
  LightGBM accuracy: 0.8278 (GPU)
  Training TensorFlow NN...
  TensorFlow accuracy: 0.8039 (GPU)
  Training RandomForest...
  RandomForest accuracy: 0.8188 (CPU)

All 3 runs complete.


## 5. Results table

In [7]:
for r in results:
    models = {'LightGBM': r['LightGBM'], 'TensorFlow': r['TensorFlow'], 'RandomForest': r['RandomForest']}
    best_model = max(models, key=models.get)
    best_acc = models[best_model]

    print(f'\n{"="*55}')
    print(f'  Dataset: {r["dataset"]}')
    print(f'{"="*55}')
    print(f'{"Model":<18} {"Accuracy":<12} {"Device":<8}  {"Best?"}')
    print('-' * 55)
    for model_name, acc in models.items():
        dev = r[f'{model_name}_device']
        is_best = '<-- BEST' if model_name == best_model else ''
        print(f'{model_name:<18} {acc:<12.4f} {dev:<8}  {is_best}')
    print(f'\n  Highest accuracy: {best_model} ({best_acc:.4f})')


  Dataset: H&M ready (median+features)
Model              Accuracy     Device    Best?
-------------------------------------------------------
LightGBM           0.8278       GPU       <-- BEST
TensorFlow         0.8039       GPU       
RandomForest       0.8188       CPU       

  Highest accuracy: LightGBM (0.8278)


In [8]:
results_df = pd.DataFrame(results)
results_df = results_df[['dataset', 'LightGBM', 'LightGBM_device',
                         'TensorFlow', 'TensorFlow_device',
                         'RandomForest', 'RandomForest_device']]
results_df.columns = ['Dataset', 'LightGBM', 'LGBM_Device',
                      'TensorFlow', 'TF_Device',
                      'RandomForest', 'RF_Device']
results_df

,Dataset,LightGBM,LGBM_Device,TensorFlow,TF_Device,RandomForest,RF_Device
0,H&M ready (median+features),0.82785,GPU,0.8039,GPU,0.818775,CPU


## Summary

| What | Details |
|------|---------|
| Dataset | H&M ready (median_version + dropped columns + 15 engineered features) |
| Algorithms | 3 (LightGBM, TensorFlow NN, RandomForest) — same settings as before |
| Total runs | 3 |
| Task | Binary classification: will customer buy article? |
| Metric | Accuracy |
| Sample size | 100K positive + 100K negative = 200K |
| Features | All numeric + categorical from engineered dataset (no price/channel to avoid leakage) |
| GPU | TensorFlow via DirectML, LightGBM tries GPU then CPU, RandomForest CPU only |